Without Memory

In [1]:
!pip -q install -U cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.7/370.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 32.0 MB/s eta 0:00:00


In [2]:
import cohere
from getpass import getpass

# 1. Connect to Cohere
api_key = getpass("Enter your Cohere API key: ")
co = cohere.ClientV2(api_key=api_key)

# 2. Agent function
def run_agent(user_input):

    # System instruction
    system_message = """
    You are a helpful AI assistant.
    Answer the user's questions clearly and accurately.
    Do not invent information.
    """

    # 3. Call Cohere
    response = co.chat(
        model="command-a-03-2025",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_input}
        ]
    )

    # 4. Extract the response
    answer = response.message.content[0].text

    return answer


# 5. Chat with the agent
while True:
    query = input("\nYou: ")

    if query.lower() == "exit":
        print("Agent: Goodbye!")
        break

    answer = run_agent(query)
    print("Agent:", answer)

Enter your Cohere API key: ··········

You: hi
Agent: Hello! How can I assist you today?

You: I am Premalatha
Agent: Hello Premalatha! It's nice to meet you. How can I assist you today? Whether you have questions, need information, or just want to chat, feel free to let me know!

You: what is my name
Agent: I don't have access to personal information, so I don't know your name. If you'd like to share it, I'd be happy to address you by it!


KeyboardInterrupt: Interrupted by user

With memory

In [3]:

import cohere
from getpass import getpass

# 1. Connect to Cohere
api_key = getpass("Enter your Cohere API key: ")
co = cohere.ClientV2(api_key=api_key)

# 2. Agent memory
memory = {}

# 3. Conversation history
history = []

# 4. Agent function
def run_agent(user_input):

    # Store information when the user shares it
    if user_input.lower().startswith("my name is "):
        name = user_input[11:].strip()
        memory["name"] = name

    if user_input.lower().startswith("i work as "):
        job = user_input[10:].strip()
        memory["job"] = job

    # Add memory to the prompt
    memory_text = str(memory)

    system_message = f"""
    You are a helpful AI assistant with memory.
    Use the following stored information when relevant:
    {memory_text}

    Answer the user's questions clearly.
    Do not invent personal information.
    """

    # Add the latest user message
    history.append({
        "role": "user",
        "content": user_input
    })

    # Call Cohere
    response = co.chat(
        model="command-a-03-2025",
        messages=[
            {"role": "system", "content": system_message},
            *history
        ]
    )

    answer = response.message.content[0].text

    # Store the assistant's response
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer


# 5. Chat with the agent
while True:
    query = input("\nYou: ")

    if query.lower() == "exit":
        print("Agent: Goodbye!")
        break

    answer = run_agent(query)
    print("Agent:", answer)


Enter your Cohere API key: ··········

You: hi i am premalatha
Agent: Hello Premalatha! It's nice to meet you. How can I assist you today?

You: what is my name
Agent: Your name is Premalatha. Is there anything else I can help you with?


KeyboardInterrupt: Interrupted by user